# Genomic Data Science - Exploratory Analysis

This notebook provides a template for exploring and analyzing genomic data with PyTorch.

## 1. Setup and Imports

In [ ]:
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
sys.path.append('..')

from src.models.base_model import BaseModel
from src.data.dataset import GenomicDataset
from src.data.dataloader import get_dataloaders
from src.utils.logger import setup_logger
from src.utils.config import load_config

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Load and Explore Data

In [ ]:
# Create dataset (using dummy data for demonstration)
dataset = GenomicDataset()

print(f"Dataset size: {len(dataset)}")
print(f"Feature dimension: {dataset.get_feature_dim()}")
print(f"Number of classes: {dataset.get_num_classes()}")

# Get a sample
features, label = dataset[0]
print(f"\nSample feature shape: {features.shape}")
print(f"Sample label: {label}")

## 3. Data Visualization

In [ ]:
# Visualize label distribution
labels = [dataset[i][1].item() for i in range(len(dataset))]

plt.figure(figsize=(8, 6))
plt.hist(labels, bins=dataset.get_num_classes(), edgecolor='black')
plt.xlabel('Class')
plt.ylabel('Frequency')
plt.title('Class Distribution')
plt.show()

## 4. Create Data Loaders

In [ ]:
# Create train/val/test splits
train_loader, val_loader, test_loader = get_dataloaders(
    dataset=dataset,
    batch_size=32,
    train_split=0.7,
    val_split=0.15,
    test_split=0.15,
    seed=42
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 5. Initialize Model

In [ ]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Initialize model
model = BaseModel(
    input_dim=dataset.get_feature_dim(),
    hidden_dim=256,
    output_dim=dataset.get_num_classes(),
    dropout=0.5
).to(device)

print(f"\nModel architecture:")
print(model)
print(f"\nTotal parameters: {model.get_num_params():,}")
print(f"Trainable parameters: {model.get_trainable_params():,}")

## 6. Quick Training Example

In [ ]:
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 5

# Training loop
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    # Training
    model.train()
    train_loss = 0.0
    
    for features, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        features, labels = features.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    
    # Validation
    model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for features, labels in val_loader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
    
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    
    print(f"Epoch {epoch+1}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

## 7. Visualize Training Progress

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Train Loss', marker='o')
plt.plot(val_losses, label='Val Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Progress')
plt.legend()
plt.grid(True)
plt.show()

## 8. Model Evaluation

In [ ]:
from src.utils.metrics import calculate_metrics, print_metrics

# Evaluate on test set
model.eval()
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for features, labels in test_loader:
        features = features.to(device)
        outputs = model(features)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

# Calculate metrics
metrics = calculate_metrics(all_labels, all_preds, all_probs)
print_metrics(metrics)

## 9. Confusion Matrix Visualization

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=metrics['confusion_matrix'])
disp.plot(ax=ax, cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

## 10. Next Steps

- Load your own genomic data
- Customize the model architecture for your specific task
- Experiment with different hyperparameters
- Add data augmentation techniques
- Implement cross-validation
- Try ensemble methods
- Add interpretability analyses (e.g., feature importance)